## CME OCT 2024

In [1]:
import pandas as pd
import numpy as np

cme_oct_2024 = pd.read_csv("../../data/raw/CME/CME_OCT_2024.csv")
cme_oct_2024 = cme_oct_2024.reset_index(drop = True)
cme_oct_2024["time"] = pd.to_datetime(cme_oct_2024["time"],unit="s") 

In [2]:
cme_oct_2024

,time,open,high,low,close,Volume
0,2024-07-01 05:39:00,94.845,94.845,94.845,94.845,150
1,2024-07-01 06:11:00,94.845,94.845,94.845,94.845,11
2,2024-07-01 06:12:00,94.845,94.845,94.845,94.845,1
3,2024-07-01 06:17:00,94.845,94.845,94.845,94.845,75
4,2024-07-01 06:18:00,94.845,94.845,94.845,94.845,26
...,...,...,...,...,...,...
25286,2024-10-31 20:05:00,95.170,95.170,95.170,95.170,2
25287,2024-10-31 20:17:00,95.170,95.170,95.170,95.170,5
25288,2024-10-31 20:38:00,95.170,95.170,95.170,95.170,1
25289,2024-10-31 20:42:00,95.170,95.170,95.170,95.170,5


## CME NOV 2024

In [3]:
cme_nov_2024 = pd.read_csv("../../data/raw/CME/CME_NOV_2024.csv")
cme_nov_2024 = cme_nov_2024.reset_index(drop = True)
cme_nov_2024["time"] = pd.to_datetime(cme_nov_2024["time"],unit="s") 

In [4]:
cme_nov_2024

,time,open,high,low,close,Volume
0,2024-08-01 05:00:00,95.0950,95.0950,95.0950,95.0950,165
1,2024-08-01 05:01:00,95.0950,95.0950,95.0950,95.0950,18
2,2024-08-01 05:02:00,95.0950,95.0950,95.0950,95.0950,22
3,2024-08-01 05:06:00,95.0950,95.0950,95.0950,95.0950,13
4,2024-08-01 05:08:00,95.0950,95.0950,95.0950,95.0950,27
...,...,...,...,...,...,...
25856,2024-11-29 14:34:00,95.3625,95.3625,95.3625,95.3625,45
25857,2024-11-29 14:51:00,95.3625,95.3625,95.3625,95.3625,13
25858,2024-11-29 15:41:00,95.3625,95.3625,95.3625,95.3625,13
25859,2024-11-29 16:25:00,95.3625,95.3625,95.3625,95.3625,3


## Merge and then calculate probabilities

In [5]:
oct = cme_oct_2024.copy()
oct["time"] = pd.to_datetime(oct["time"], unit="s")
oct = oct.sort_values("time").set_index("time")

oct = oct.add_suffix("_oct_2024")

nov = cme_nov_2024.copy()
nov["time"] = pd.to_datetime(nov["time"], unit="s")
nov = nov.sort_values("time").set_index("time")

nov = nov.add_suffix("_nov_2024")

oct_aligned = oct.reindex(nov.index, method="ffill")


final_df = pd.concat([nov, oct_aligned], axis=1)
final_df = final_df.reset_index()

In [6]:
final_df

,time,open_nov_2024,high_nov_2024,low_nov_2024,close_nov_2024,Volume_nov_2024,open_oct_2024,high_oct_2024,low_oct_2024,close_oct_2024,Volume_oct_2024
0,2024-08-01 05:00:00,95.0950,95.0950,95.0950,95.0950,165,94.95,94.95,94.95,94.95,21
1,2024-08-01 05:01:00,95.0950,95.0950,95.0950,95.0950,18,94.95,94.95,94.95,94.95,21
2,2024-08-01 05:02:00,95.0950,95.0950,95.0950,95.0950,22,94.95,94.95,94.95,94.95,21
3,2024-08-01 05:06:00,95.0950,95.0950,95.0950,95.0950,13,94.95,94.95,94.95,94.95,21
4,2024-08-01 05:08:00,95.0950,95.0950,95.0950,95.0950,27,94.95,94.95,94.95,94.95,21
...,...,...,...,...,...,...,...,...,...,...,...
25856,2024-11-29 14:34:00,95.3625,95.3625,95.3625,95.3625,45,95.17,95.17,95.17,95.17,13
25857,2024-11-29 14:51:00,95.3625,95.3625,95.3625,95.3625,13,95.17,95.17,95.17,95.17,13
25858,2024-11-29 15:41:00,95.3625,95.3625,95.3625,95.3625,13,95.17,95.17,95.17,95.17,13
25859,2024-11-29 16:25:00,95.3625,95.3625,95.3625,95.3625,3,95.17,95.17,95.17,95.17,13


## Calculating Implied Probabilities

In [7]:
meeting_day = 7
days_in_nov = 30

N = meeting_day           
M = days_in_nov - N        

imp_df = pd.DataFrame()
imp_df["time"] = final_df["time"].copy()

imp_df["effr_avg_oct"] = 100 - final_df["close_oct_2024"]
imp_df["effr_avg_nov"] = 100 - final_df["close_nov_2024"]

imp_df["effr_end_oct"] = imp_df["effr_avg_oct"]
imp_df["effr_start_nov"] = imp_df["effr_end_oct"]


In [8]:
imp_df

,time,effr_avg_oct,effr_avg_nov,effr_end_oct,effr_start_nov
0,2024-08-01 05:00:00,5.05,4.9050,5.05,5.05
1,2024-08-01 05:01:00,5.05,4.9050,5.05,5.05
2,2024-08-01 05:02:00,5.05,4.9050,5.05,5.05
3,2024-08-01 05:06:00,5.05,4.9050,5.05,5.05
4,2024-08-01 05:08:00,5.05,4.9050,5.05,5.05
...,...,...,...,...,...
25856,2024-11-29 14:34:00,4.83,4.6375,4.83,4.83
25857,2024-11-29 14:51:00,4.83,4.6375,4.83,4.83
25858,2024-11-29 15:41:00,4.83,4.6375,4.83,4.83
25859,2024-11-29 16:25:00,4.83,4.6375,4.83,4.83


In [9]:
imp_df["effr_end_nov"] = (
    imp_df["effr_avg_nov"] - (N / (N + M)) * imp_df["effr_start_nov"]
) / (M / (N + M))

In [10]:
imp_df["delta_effr"] = imp_df["effr_end_nov"] - imp_df["effr_start_nov"]
imp_df["num_hikes"] = imp_df["delta_effr"] / 0.25

imp_df["hikes_floor"] = np.floor(imp_df["num_hikes"])
imp_df["hikes_decimal"] = imp_df["num_hikes"] - imp_df["hikes_floor"]

states = [-2, -1, 0, 1, 2]

state_to_col = {
    -2: "prob_cut_50",
    -1: "prob_cut_25",
     0: "prob_no_change",
     1: "prob_hike_25",
     2: "prob_hike_50"
}

for col in state_to_col.values():
    imp_df[col] = 0.0

for i, row in imp_df.iterrows():
    floor = int(row["hikes_floor"])
    decimal = row["hikes_decimal"]
    
    p_low = 1 - decimal
    p_high = decimal
    
    if floor in state_to_col:
        imp_df.at[i, state_to_col[floor]] += p_low
    
    if (floor + 1) in state_to_col:
        imp_df.at[i, state_to_col[floor + 1]] += p_high

In [11]:
imp_df

,time,effr_avg_oct,effr_avg_nov,effr_end_oct,effr_start_nov,effr_end_nov,delta_effr,num_hikes,hikes_floor,hikes_decimal,prob_cut_50,prob_cut_25,prob_no_change,prob_hike_25,prob_hike_50
0,2024-08-01 05:00:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
1,2024-08-01 05:01:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
2,2024-08-01 05:02:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
3,2024-08-01 05:06:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
4,2024-08-01 05:08:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25856,2024-11-29 14:34:00,4.83,4.6375,4.83,4.83,4.578913,-0.251087,-1.004348,-2.0,0.995652,0.004348,0.995652,0.000000,0.0,0.0
25857,2024-11-29 14:51:00,4.83,4.6375,4.83,4.83,4.578913,-0.251087,-1.004348,-2.0,0.995652,0.004348,0.995652,0.000000,0.0,0.0
25858,2024-11-29 15:41:00,4.83,4.6375,4.83,4.83,4.578913,-0.251087,-1.004348,-2.0,0.995652,0.004348,0.995652,0.000000,0.0,0.0
25859,2024-11-29 16:25:00,4.83,4.6375,4.83,4.83,4.578913,-0.251087,-1.004348,-2.0,0.995652,0.004348,0.995652,0.000000,0.0,0.0


In [12]:
imp_df["time"] = pd.to_datetime(imp_df["time"], utc=True)

## Filtering Based on Announcement Time

In [13]:
fomc_2024_utc = {
    "January 2024": pd.Timestamp("2024-01-31 19:00:00", tz="UTC"),   
    "March 2024": pd.Timestamp("2024-03-20 18:00:00", tz="UTC"),     
    "May 2024": pd.Timestamp("2024-05-01 18:00:00", tz="UTC"),       
    "June 2024": pd.Timestamp("2024-06-12 18:00:00", tz="UTC"),      
    "July 2024": pd.Timestamp("2024-07-31 18:00:00", tz="UTC"),      
    "novtember 2024": pd.Timestamp("2024-09-18 18:00:00", tz="UTC"), 
    "November 2024": pd.Timestamp("2024-11-07 19:00:00", tz="UTC"),  
    "December 2024": pd.Timestamp("2024-12-18 19:00:00", tz="UTC"),  
}

In [14]:
def filter_up_to_announcement(df, announcement_dict):

    if df.empty:
        return df
    last_timestamp = df["time"].iloc[-1]
    last_month_year = last_timestamp.strftime("%B %Y")

    if last_month_year in announcement_dict:
        cutoff_time = announcement_dict[last_month_year]
        filtered_df = df[df["time"] <= cutoff_time]
        return filtered_df
    else:
        print(
            f"Warning: '{last_month_year}' not found in the announcement dictionary"
        )
        return df

In [15]:
imp_df_fil = filter_up_to_announcement(imp_df,fomc_2024_utc)

In [16]:
imp_df_fil

,time,effr_avg_oct,effr_avg_nov,effr_end_oct,effr_start_nov,effr_end_nov,delta_effr,num_hikes,hikes_floor,hikes_decimal,prob_cut_50,prob_cut_25,prob_no_change,prob_hike_25,prob_hike_50
0,2024-08-01 05:00:00+00:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
1,2024-08-01 05:01:00+00:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
2,2024-08-01 05:02:00+00:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
3,2024-08-01 05:06:00+00:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
4,2024-08-01 05:08:00+00:00,5.05,4.9050,5.05,5.05,4.860870,-0.189130,-0.756522,-1.0,0.243478,0.000000,0.756522,0.243478,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24968,2024-11-07 18:56:00+00:00,4.83,4.6375,4.83,4.83,4.578913,-0.251087,-1.004348,-2.0,0.995652,0.004348,0.995652,0.000000,0.0,0.0
24969,2024-11-07 18:57:00+00:00,4.83,4.6375,4.83,4.83,4.578913,-0.251087,-1.004348,-2.0,0.995652,0.004348,0.995652,0.000000,0.0,0.0
24970,2024-11-07 18:58:00+00:00,4.83,4.6375,4.83,4.83,4.578913,-0.251087,-1.004348,-2.0,0.995652,0.004348,0.995652,0.000000,0.0,0.0
24971,2024-11-07 18:59:00+00:00,4.83,4.6375,4.83,4.83,4.578913,-0.251087,-1.004348,-2.0,0.995652,0.004348,0.995652,0.000000,0.0,0.0


## Export the Data as a CSV

In [17]:
import os
DATA_DIR = "../../data/processed/CME_implied_probabilities"
filename = "CME_IMP_NOV_2024.csv"
path = os.path.join(DATA_DIR, filename)
imp_df_fil.to_csv(path,index=False)